# PROJECT 1 · STEP 6 — Root Cause Verification & Robust Optimization

평균 최적점이 아니라 MSA `+3σ` guard band를 적용한 multi-CTQ process window를 구한다.

In [ ]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
boot = pd.read_csv(ROOT / 'results/bootstrap_effects.csv')
window = pd.read_csv(ROOT / 'results/robust_process_window.csv')
verification = pd.read_csv(ROOT / 'results/root_cause_verification_matrix.csv')
summary = json.loads((ROOT / 'results/verification_summary.json').read_text(encoding='utf-8'))
summary

## 1. Effect uncertainty

Process-factor effect는 각 whole plot 내부 high−low contrast를 계산한 뒤 whole plot 단위 bootstrap을 사용한다. Block 수가 4개뿐이므로 CI는 탐색적으로 해석한다.

In [ ]:
boot.query("response == 'edge_void_pct'")[['term','mean_effect','ci95_low','ci95_high','direction_stable']]

## 2. Robust pass

`Predicted Void + 3σGRR ≤ 0.50%`, `Predicted Offset + 3σGRR ≤ 20 μm`, Warpage와 Cycle Time 제약을 동시에 만족해야 한다.

In [ ]:
window.query("emc_lot == 'M02' and film_roughness_class == 'Smooth' and robust_pass")[[
    'vacuum_base_kpa_abs','zone_range_c','closing_speed_mm_s',
    'void_upper_3sigma_msa','offset_upper_3sigma_msa','warpage_um','cycle_time_index'
]]

## 3. Causal claim boundary

H1은 synthetic DOE에서 조작 evidence가 있다. H2는 associative, H3/H5는 unreplicated whole-plot, H4는 미조작이므로 같은 강도로 주장하지 않는다.

In [ ]:
verification[['claim','decision','remaining_real_test']]